# Lesson 9: Build one causal self-attention head

Karpathy now turns the earlier attention math into a reusable Head module. For each token, three learned linear layers produce a **query**, **key**, and **value**. Query-key dot products decide which earlier values to combine.

The example is deliberately tiny: batch size 2, sequence length 4, input width 4, and head size 2. We set the projection weights to small fixed values so the Python, C, and Java results are directly comparable. In real training, these weights start from initialization and are updated by gradient descent.

The matching [C implementation](../c/src/attention_head_main.c), [Java implementation](../java/src/main/java/learning/karpathy/gpt1/CausalAttentionHeadLesson.java), and [shared lesson map](../README.md#run-lessons-by-step) use the same inputs and weights.

This follows the self-attention and scaling examples around cells 23–26 in the original [gpt_dev notebook](../gpt_dev.ipynb). Karpathy's Head also applies dropout to attention weights during training. We leave dropout out here so all three languages stay deterministic; it is a useful follow-up once the forward pass is clear.

## Set up the input and the head module

The input shape is (batch, time, channels). The three nn.Linear layers keep batch and time dimensions and map the final channel dimension from 4 to 2. The triangular mask is registered as a module buffer: it belongs with the layer but is not a trainable parameter.

In [ ]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.set_printoptions(precision=4, sci_mode=False)

BATCH_SIZE = 2
TIME_STEPS = 4
CHANNELS = 4
HEAD_SIZE = 2

x = torch.tensor(
    [
        [
            [1.0, 0.0, 2.0, -1.0],
            [0.0, 1.0, 1.0, 2.0],
            [2.0, -1.0, 0.0, 1.0],
            [1.0, 2.0, -1.0, 0.0],
        ],
        [
            [0.0, 2.0, 1.0, -1.0],
            [1.0, -1.0, 2.0, 0.0],
            [-1.0, 0.0, 1.0, 2.0],
            [2.0, 1.0, 0.0, -1.0],
        ],
    ],
    dtype=torch.float64,
)

class CausalAttentionHead(nn.Module):
    def __init__(self, channels: int, head_size: int, max_time: int) -> None:
        super().__init__()
        self.query = nn.Linear(channels, head_size, bias=False, dtype=torch.float64)
        self.key = nn.Linear(channels, head_size, bias=False, dtype=torch.float64)
        self.value = nn.Linear(channels, head_size, bias=False, dtype=torch.float64)
        causal_mask = torch.tril(torch.ones(max_time, max_time, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        batch, time, _channels = inputs.shape
        queries = self.query(inputs)
        keys = self.key(inputs)
        values = self.value(inputs)

        scores = queries @ keys.transpose(-2, -1) / math.sqrt(queries.shape[-1])
        scores = scores.masked_fill(~self.causal_mask[:time, :time], float("-inf"))
        weights = F.softmax(scores, dim=-1)
        return weights @ values

head = CausalAttentionHead(CHANNELS, HEAD_SIZE, TIME_STEPS)

# Copy small hand-chosen weights into Linear.weight [out_features, in_features].
# Fixed values make it easy to compare all three language versions.
with torch.no_grad():
    head.query.weight.copy_(torch.tensor(
        [[0.5, -0.5, 1.0, 0.0], [0.0, 1.0, 0.5, -0.5]], dtype=torch.float64
    ))
    head.key.weight.copy_(torch.tensor(
        [[1.0, 0.0, 0.5, -0.5], [-0.5, 0.5, 0.0, 1.0]], dtype=torch.float64
    ))
    head.value.weight.copy_(torch.tensor(
        [[0.5, 0.0, 1.0, 0.5], [0.0, 1.0, -0.5, 0.5]], dtype=torch.float64
    ))

print("Input shape [batch, time, channels]:", tuple(x.shape))
print(head)

## Step 1: Project inputs into queries, keys, and values

Each token vector gets three different views. A query asks what information this token needs; a key describes what a token offers for matching; a value carries the content that can be mixed into the output. All three use learned linear projections in the full model.

The matrix convention is output = input @ weight.T, matching PyTorch Linear's stored weight shape (out_features, in_features).

In [ ]:
queries = head.query(x)
keys = head.key(x)
values = head.value(x)

print("Shapes: Q", tuple(queries.shape), "K", tuple(keys.shape), "V", tuple(values.shape))
print("Queries for batch 0:\n", queries[0])
print("Keys for batch 0:\n", keys[0])
print("Values for batch 0:\n", values[0])

## Step 2: Score every query-key pair and scale

For each batch, the matrix product Q @ K.transpose(-2, -1) compares each query with every key. Its shape is (batch, query time, key time). Dividing by sqrt(head_size) keeps the score scale from growing with the vector width, as in Lesson 7.

In [ ]:
raw_scores = queries @ keys.transpose(-2, -1)
scaled_scores = raw_scores / math.sqrt(HEAD_SIZE)

print("Score shape [batch, query time, key time]:", tuple(scaled_scores.shape))
print("Raw scores for batch 0:\n", raw_scores[0])
print("Scaled scores for batch 0:\n", scaled_scores[0])

## Step 3: Mask future tokens, then apply softmax

A decoder must not use tokens that come later in the sequence. The lower-triangular boolean mask keeps each position's own token and its past. Replacing future scores with negative infinity makes their softmax weights exactly zero. Softmax runs across the key-time dimension, so each query's row becomes a probability distribution.

In [ ]:
causal_mask = head.causal_mask[:TIME_STEPS, :TIME_STEPS]
masked_scores = scaled_scores.masked_fill(~causal_mask, float("-inf"))
attention_weights = F.softmax(masked_scores, dim=-1)

print("Causal mask:\n", causal_mask)
print("Masked scores for batch 0:\n", masked_scores[0])
print("Attention weights for batch 0:\n", attention_weights[0])
print("Row sums:", attention_weights.sum(dim=-1))
assert torch.allclose(attention_weights.sum(dim=-1), torch.ones(BATCH_SIZE, TIME_STEPS, dtype=attention_weights.dtype))
assert torch.count_nonzero(attention_weights[:, ~causal_mask]) == 0

## Step 4: Mix values and verify causality

Each output is a weighted sum of value vectors. Its last dimension is the head size (2), while batch and time are preserved. Finally, change a future input token by a large amount: outputs at earlier positions must remain exactly the same because their rows cannot attend to that future position.

In [ ]:
output = attention_weights @ values
module_output = head(x)

print("Output shape [batch, time, head_size]:", tuple(output.shape))
print("Single-head output:\n", output[0])
torch.testing.assert_close(module_output, output)

changed_x = x.clone()
changed_x[0, 3, 0] += 100.0
changed_output = head(changed_x)

earlier_difference = (output[0, :3] - changed_output[0, :3]).abs().max()
edited_position_difference = (output[0, 3] - changed_output[0, 3]).abs().max()
print(f"Change at earlier positions: {earlier_difference.item():.12f}")
print(f"Change at edited position:   {edited_position_difference.item():.12f}")

torch.testing.assert_close(output[0, :3], changed_output[0, :3], rtol=0, atol=1e-12)
assert edited_position_difference > 1e-8
print("Causality check passed: future input changes cannot alter earlier outputs.")

## What to notice

The mask is what makes this a *causal* head; query-key similarity alone can compare every position with every other position. The output is narrower than the input here because one attention head writes head_size channels. A multi-head block will concatenate several such outputs and project them back to the model width.

In the C and Java companions, the same calculations are explicit loops over arrays. Their weights are fixed constants, so they explain the forward pass but do not train parameters. In this Python version the nn.Linear weights are parameters and can receive gradients, although this lesson does not run an optimizer.

References: [PyTorch Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html), [PyTorch Module buffers](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer), [PyTorch softmax](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html), and Karpathy's original [GPT from scratch notebook](../gpt_dev.ipynb).